# Stage 0 — Setup

One-time setup for the workshop. Run every cell top-to-bottom.

You'll install Python + Node dependencies, bridge your OpenAI key into
the environment, and stage the bundle-shipped config files into the
working directories each stage expects.


## 1. Clone the repo


In [ ]:
# Bootstrap: clone the workshop repo into /content and cd into it.
# Idempotent — safe to re-run.
import os, subprocess, sys
REPO_DIR = "/content/ar-bic-2026-workshop"
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/jayprimer/ar-bic-2026-workshop.git", REPO_DIR],
        check=True,
    )
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())


## 2. Install dependencies

Python: `openai`. Node: `@llamaindex/liteparse` (Stage 4's PDF→text CLI).


In [ ]:
!pip install -q -r requirements.txt


In [ ]:
# liteparse is a Node CLI, not a Python lib. Stage 4 invokes it via subprocess.
!npm install -g @llamaindex/liteparse 2>&1 | tail -5
!lit --version || echo 'lit not on PATH — check the install output above'


## 3. Bridge your OpenAI key

Add `OPENAI_API_KEY` in **Colab Secrets** (key icon in the left sidebar) and toggle notebook access, then run this cell.


In [ ]:
# OpenAI key bridge: Colab's userdata.get() does NOT populate os.environ,
# but our scripts read os.environ["OPENAI_API_KEY"]. Bridge it once.
# Add the key in Colab via the left sidebar → "Secrets" (key icon) → name it OPENAI_API_KEY.
import os
try:
    from google.colab import userdata
    key = userdata.get("OPENAI_API_KEY")
    if key:
        os.environ["OPENAI_API_KEY"] = key
        print("OPENAI_API_KEY set in os.environ")
    else:
        print("WARNING: OPENAI_API_KEY secret is empty — Stage 2/5 and *_llm.py evals will fail")
except Exception as e:
    print("Not running in Colab or userdata unavailable; set OPENAI_API_KEY yourself.")
    print("Detail:", e)


## 4. Stage the config files

Each stage script reads `stage_NN/input.txt` (and `stage_02/criteria.txt`) relative to the current working directory. Copy the bundle-shipped configs into place so participant scripts find them.


In [ ]:
import os, shutil
os.makedirs("stage_01", exist_ok=True)
os.makedirs("stage_02", exist_ok=True)
shutil.copy("configs/stage_01_input.txt",    "stage_01/input.txt")
shutil.copy("configs/stage_02_input.txt",    "stage_02/input.txt")
shutil.copy("configs/stage_02_criteria.txt", "stage_02/criteria.txt")
# Stage 5 reads schema.json from cwd, so put a copy at the repo root too.
shutil.copy("configs/schema.json", "schema.json")
print("staged:", os.listdir("stage_01"), os.listdir("stage_02"))


## 5. You're ready

Open `notebooks/stage_01_search.ipynb` next.

If anything failed above, re-run the failing cell — most install hiccups
clear on retry. Network errors hitting NCBI later in the workshop are
also retry-friendly.
